# Renderização de Animações do Unreal Engine 4 no Google Colab GPU T4

Este notebook configura um ambiente para renderizar animações do Unreal Engine 4 usando a GPU T4 do Google Colab.

**⚠️ Importante:**
- Certifique-se de ter ativado o runtime GPU T4 (Runtime → Change runtime type → GPU)
- O Unreal Engine requer recursos significativos, então use com cautela
- Este notebook é para renderização offline, não para desenvolvimento interativo

## 1. Verificar GPU e Configuração do Sistema

In [ ]:
# Verificar se a GPU está disponível
!nvidia-smi

# Verificar espaço em disco
!df -h

# Verificar memória RAM
!free -h

## 2. Instalar Dependências Necessárias

In [ ]:
# Atualizar sistema e instalar dependências básicas
!apt-get update
!apt-get install -y wget unzip libvulkan1 vulkan-utils mesa-vulkan-drivers \
    libgl1-mesa-glx libglu1-mesa libxrandr2 libxi6 libxcursor1 libxinerama1 \
    libxrender1 libfontconfig1 libxext6 libsm6 libice6 libxau6 libxdmcp6

## 3. Configurar Virtual Display (Xvfb) para Renderização Headless

In [ ]:
# Instalar Xvfb para renderização sem interface gráfica
!apt-get install -y xvfb x11-utils

# Iniciar display virtual
import os
os.system('Xvfb :1 -screen 0 1920x1080x24 &')
os.environ['DISPLAY'] = ':1'

print("Display virtual configurado!")

## 4. Montar Google Drive (para acessar seus arquivos do UE4)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Definir caminhos
DRIVE_PATH = '/content/drive/MyDrive'
UE4_PROJECT_PATH = f'{DRIVE_PATH}/UE4_Projects'  # Ajuste conforme necessário
OUTPUT_PATH = f'{DRIVE_PATH}/UE4_Renders'

# Criar pasta de output se não existir
!mkdir -p {OUTPUT_PATH}

print(f"Google Drive montado!")
print(f"Projetos: {UE4_PROJECT_PATH}")
print(f"Renders: {OUTPUT_PATH}")

## 5. Configurar Token do GitHub e Clonar Unreal Engine 4.27 (Chaos Branch)

**Nota:** O Unreal Engine source code requer:
1. Conta GitHub vinculada ao Epic Games
2. Token GitHub com acesso ao repositório EpicGames/UnrealEngine
3. Compilação do engine (~2-3 horas na primeira vez)
4. Espaço em disco: ~50-60GB após compilação

**Como obter acesso:**
1. Vincule sua conta GitHub à Epic Games: https://www.unrealengine.com/account/connections
2. Aceite o convite do repositório EpicGames/UnrealEngine
3. Crie um Personal Access Token no GitHub com permissões `repo`

In [ ]:
# CONFIGURAÇÃO DO TOKEN GITHUB
import getpass
from google.colab import userdata
import os

print("=" * 60)
print("CONFIGURAÇÃO DO GITHUB TOKEN")
print("=" * 60)
print("\n📋 Passos para obter o token:")
print("1. Vincule GitHub à Epic Games: https://www.unrealengine.com/account/connections")
print("2. Aceite convite do repo: https://github.com/EpicGames/UnrealEngine")
print("3. Crie token: GitHub → Settings → Developer settings → Personal access tokens")
print("4. Permissões necessárias: 'repo' (acesso completo)")
print("\n" + "=" * 60)

# Solicitar token (será armazenado de forma segura)
GITHUB_TOKEN = getpass.getpass('\n🔑 Cole seu GitHub Personal Access Token: ')

if not GITHUB_TOKEN:
    raise ValueError("❌ Token do GitHub é obrigatório!")

# Configurar Git credentials
os.system(f'git config --global credential.helper store')
os.system(f'git config --global user.name "Colab User"')
os.system(f'git config --global user.email "user@colab.com"')

print("\n✅ Token configurado com sucesso!")

# Definir diretórios
UE4_SOURCE_DIR = '/content/UnrealEngine'
UE4_BRANCH = '4.27-chaos'

print(f"\n📁 Diretório UE4: {UE4_SOURCE_DIR}")
print(f"🌿 Branch: {UE4_BRANCH}")

In [ ]:
# ALTERNATIVA: Baixar engine pré-compilado do Google Drive
# Descomente e ajuste se você tem um build pré-compilado

USE_PRECOMPILED = False  # Mude para True se tiver build pré-compilado

if USE_PRECOMPILED:
    print("📦 Usando engine pré-compilado do Google Drive...")
    
    PRECOMPILED_PATH = f'{DRIVE_PATH}/UE4_Linux_Compiled.tar.gz'
    
    if os.path.exists(PRECOMPILED_PATH):
        print(f"🔽 Extraindo {PRECOMPILED_PATH}...")
        !tar -xzf {PRECOMPILED_PATH} -C /content/
        
        # Atualizar caminho do executável
        UE4_EXECUTABLE = '/content/UnrealEngine/Engine/Binaries/Linux/UnrealEditor'
        
        if os.path.exists(UE4_EXECUTABLE):
            print(f"✅ Engine pré-compilado extraído com sucesso!")
            print(f"🎮 Executável: {UE4_EXECUTABLE}")
        else:
            print("❌ Executável não encontrado após extração")
    else:
        print(f"❌ Arquivo não encontrado: {PRECOMPILED_PATH}")
        print("Você precisa fazer upload do engine compilado para o Drive")
else:
    # Definir executável da compilação normal
    UE4_EXECUTABLE = f'{UE4_SOURCE_DIR}/Engine/Binaries/Linux/UnrealEditor'
    print(f"✅ Usando executável compilado: {UE4_EXECUTABLE}")

# Verificar se executável existe e tem permissão
if os.path.exists(UE4_EXECUTABLE):
    !chmod +x {UE4_EXECUTABLE}
    print(f"\n🎮 Unreal Engine pronto para uso!")
    
    # Testar versão
    print("\n📋 Versão do engine:")
    !{UE4_EXECUTABLE} -version 2>&1 | head -n 5 || echo "Comando -version não suportado"
else:
    print(f"\n⚠️  AVISO: Executável não encontrado em {UE4_EXECUTABLE}")
    print("Execute as células de compilação acima primeiro!")

## 5.6. (Alternativa Rápida) Usar Projeto Pré-compilado do Drive

Se a compilação for muito demorada, você pode:
1. Compilar localmente no seu PC Linux
2. Empacotar o engine compilado
3. Fazer upload para o Google Drive
4. Baixar aqui no Colab (muito mais rápido)

In [ ]:
import multiprocessing

print("=" * 60)
print("COMPILANDO UNREAL ENGINE 4.27")
print("=" * 60)

os.chdir(UE4_SOURCE_DIR)

# Número de cores para compilação
num_cores = multiprocessing.cpu_count()
print(f"🔧 CPU Cores disponíveis: {num_cores}")
print(f"⚠️  Compilação pode levar 2-4 horas")
print(f"💾 Monitorando progresso...\n")

start_time = time.time()

# Compilar apenas os targets necessários para renderização
# UnrealEditor: Editor do engine (necessário para abrir projetos)
# ShaderCompileWorker: Compilação de shaders
print("🔨 Compilando UnrealEditor (Development)...\n")

compile_cmd = [
    'make',
    'UnrealEditor',
    f'-j{num_cores}',  # Parallel jobs
    'ARGS=-Progress'
]

try:
    # Executar compilação com output em tempo real
    process = subprocess.Popen(
        compile_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        cwd=UE4_SOURCE_DIR
    )
    
    # Mostrar progresso
    for line in process.stdout:
        # Filtrar apenas linhas importantes
        if any(keyword in line for keyword in ['[', '%', 'Compiling', 'Linking', 'Error', 'Warning']):
            print(line, end='')
    
    process.wait()
    
    if process.returncode == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"\n✅ COMPILAÇÃO CONCLUÍDA em {elapsed:.1f} minutos!")
        
        # Verificar executável
        UE4_EXECUTABLE = f'{UE4_SOURCE_DIR}/Engine/Binaries/Linux/UnrealEditor'
        if os.path.exists(UE4_EXECUTABLE):
            print(f"\n🎮 Executável criado: {UE4_EXECUTABLE}")
            !ls -lh {UE4_EXECUTABLE}
        else:
            print(f"❌ Executável não encontrado em {UE4_EXECUTABLE}")
    else:
        print(f"\n❌ Erro na compilação (código {process.returncode})")
        
except KeyboardInterrupt:
    print("\n⚠️  Compilação interrompida pelo usuário")
    process.terminate()
except Exception as e:
    print(f"\n❌ Erro durante compilação: {str(e)}")

# Estatísticas finais
print("\n📊 Estatísticas:")
!du -sh {UE4_SOURCE_DIR}
!du -sh {UE4_SOURCE_DIR}/Engine/Binaries

## 5.5. Compilar Unreal Engine (Editor Headless)

**⚠️ ATENÇÃO:**
- Compilação completa pode levar **2-4 horas** no Colab
- Usa **muito** CPU e RAM
- Pode ser interrompida pelo timeout do Colab (12h)
- Para renderização, precisamos apenas do **UnrealEditor** e **UnrealPak**

In [ ]:
print("=" * 60)
print("GERANDO ARQUIVOS DE PROJETO")
print("=" * 60)

os.chdir(UE4_SOURCE_DIR)

# Gerar makefiles
print("\n⚙️  Gerando makefiles...\n")
!./GenerateProjectFiles.sh

print("\n✅ Arquivos de projeto gerados!")

# Listar estrutura principal
print("\n📁 Estrutura do projeto:")
!ls -lh {UE4_SOURCE_DIR} | head -n 20

## 5.4. Gerar Arquivos de Projeto

In [ ]:
print("=" * 60)
print("EXECUTANDO SETUP.SH")
print("=" * 60)
print("⚠️  Isso baixará dependências adicionais (~10-15 minutos)\n")

os.chdir(UE4_SOURCE_DIR)

# Dar permissão de execução aos scripts
!chmod +x Setup.sh
!chmod +x GenerateProjectFiles.sh

# Executar Setup.sh (baixa dependências binárias)
print("🔽 Baixando dependências do engine...\n")
!./Setup.sh

print("\n✅ Setup concluído!")

# Verificar espaço em disco após setup
print("\n💾 Espaço em disco usado:")
!du -sh {UE4_SOURCE_DIR}

## 5.3. Executar Setup do Unreal Engine

In [ ]:
print("=" * 60)
print("INSTALANDO DEPENDÊNCIAS DE COMPILAÇÃO")
print("=" * 60)

# Atualizar sistema
print("\n📦 Atualizando sistema...\n")
!apt-get update -qq

# Instalar ferramentas de compilação
print("\n🔧 Instalando compiladores e ferramentas...\n")
!apt-get install -y -qq \
    build-essential \
    clang \
    cmake \
    mono-mcs \
    mono-devel \
    mono-xbuild \
    dos2unix \
    libfreetype6-dev \
    libgtk-3-dev \
    libmono-microsoft-build-tasks-v4.0-4.0-cil \
    xdg-user-dirs

# Instalar dependências específicas do UE4
print("\n📚 Instalando bibliotecas do UE4...\n")
!apt-get install -y -qq \
    libsdl2-dev \
    libicu-dev \
    libogg-dev \
    libvorbis-dev \
    libopus-dev \
    libpng-dev \
    libjpeg-dev \
    libxcb-xinerama0 \
    libxcb-xinput0

print("\n✅ Dependências instaladas!")

# Verificar versões
print("\n📋 Versões instaladas:")
!clang --version | head -n 1
!cmake --version | head -n 1
!mono --version | head -n 1

## 5.2. Instalar Dependências de Compilação

In [ ]:
import subprocess
import time

# URL do repositório com autenticação
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/EpicGames/UnrealEngine.git'

print("=" * 60)
print("CLONANDO UNREAL ENGINE 4.27-CHAOS")
print("=" * 60)
print(f"⚠️  Isso pode levar 30-60 minutos (~15GB download)")
print(f"📦 Branch: {UE4_BRANCH}\n")

start_time = time.time()

# Clonar repositório (shallow clone para economizar espaço e tempo)
if not os.path.exists(UE4_SOURCE_DIR):
    print("🔽 Iniciando clone do repositório...\n")
    
    clone_cmd = [
        'git', 'clone',
        '--branch', UE4_BRANCH,
        '--depth', '1',  # Shallow clone (apenas último commit)
        '--single-branch',
        REPO_URL,
        UE4_SOURCE_DIR
    ]
    
    try:
        result = subprocess.run(
            clone_cmd,
            capture_output=True,
            text=True,
            timeout=3600  # 1 hora timeout
        )
        
        if result.returncode == 0:
            elapsed = (time.time() - start_time) / 60
            print(f"\n✅ Clone concluído em {elapsed:.1f} minutos!")
        else:
            print(f"❌ Erro no clone: {result.stderr}")
            raise Exception("Falha ao clonar repositório")
            
    except subprocess.TimeoutExpired:
        print("⏱️ Timeout - Clone demorou mais de 1 hora")
        raise
else:
    print(f"ℹ️  Repositório já existe em {UE4_SOURCE_DIR}")
    print("Atualizando...")
    os.chdir(UE4_SOURCE_DIR)
    !git pull origin {UE4_BRANCH}

# Verificar clone
os.chdir(UE4_SOURCE_DIR)
print(f"\n📊 Estatísticas do repositório:")
!du -sh {UE4_SOURCE_DIR}
!git log -1 --oneline
!git branch

## 5.1. Clonar Repositório do Unreal Engine 4.27

## 6. Função Helper para Renderização de Sequências

In [ ]:
import subprocess
import os
from pathlib import Path

def render_ue4_sequence(
    project_path,
    sequence_path,
    output_path,
    resolution='1920x1080',
    frame_rate=30,
    output_format='png',
    quality=100,
    use_movie_render_queue=True
):
    """
    Renderiza uma sequência do Unreal Engine 4.27
    
    Args:
        project_path: Caminho para o arquivo .uproject
        sequence_path: Caminho da sequência dentro do projeto (ex: /Game/Cinematics/MainSequence)
        output_path: Pasta onde salvar os frames renderizados
        resolution: Resolução do render (WIDTHxHEIGHT)
        frame_rate: Taxa de quadros
        output_format: Formato de saída (png, jpg, exr)
        quality: Qualidade de renderização (0-100)
        use_movie_render_queue: Usar Movie Render Queue (mais recursos) ou Legacy Sequencer
    """
    
    if not os.path.exists(UE4_EXECUTABLE):
        print(f"❌ Executável do UE4 não encontrado: {UE4_EXECUTABLE}")
        print("Execute as células de compilação primeiro!")
        return False
    
    # Criar diretório de output
    os.makedirs(output_path, exist_ok=True)
    
    # Comando base
    cmd = [
        UE4_EXECUTABLE,
        project_path,
        '-game',
        '-NoLoadingScreen',
        '-ForceRes',
        '-Windowed',
        '-NoScreenMessages',
        '-NoTextureStreaming',
        '-AllowStandardShaderCompile',
        '-NOTEXTURESTREAMING'
    ]
    
    if use_movie_render_queue:
        # Movie Render Queue (UE 4.27+) - Melhor qualidade
        cmd.extend([
            '-MoviePipelineConfig=/Game/MovieRenderQueue/YourConfig',  # Ajustar path do config
            '-LevelSequence=' + sequence_path,
            '-MovieSceneCaptureType=/Script/MovieSceneCapture.MoviePipelineAutomatedCapture'
        ])
    else:
        # Legacy Sequencer Render
        cmd.extend([
            '-MovieSceneCaptureType=/Script/MovieSceneCapture.AutomatedLevelSequenceCapture',
            '-LevelSequence=' + sequence_path,
            '-MovieFolder=' + output_path,
            '-ResX=' + resolution.split("x")[0],
            '-ResY=' + resolution.split("x")[1],
            '-MovieFormat=' + output_format.upper(),
            '-MovieQuality=' + str(quality),
            '-MovieFrameRate=' + str(frame_rate)
        ])
    
    print("=" * 70)
    print("🎬 INICIANDO RENDERIZAÇÃO UNREAL ENGINE 4.27")
    print("=" * 70)
    print(f"📁 Projeto: {project_path}")
    print(f"🎞️  Sequência: {sequence_path}")
    print(f"💾 Output: {output_path}")
    print(f"📐 Resolução: {resolution} @ {frame_rate}fps")
    print(f"🎨 Formato: {output_format.upper()} (Qualidade: {quality})")
    print(f"⚙️  Método: {'Movie Render Queue' if use_movie_render_queue else 'Legacy Sequencer'}")
    print("=" * 70 + "\n")
    
    try:
        # Executar com output em tempo real
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True
        )
        
        # Mostrar progresso
        for line in process.stdout:
            print(line, end='')
        
        process.wait()
        
        if process.returncode == 0:
            print("\n" + "=" * 70)
            print("✅ RENDERIZAÇÃO CONCLUÍDA COM SUCESSO!")
            print("=" * 70)
            print(f"📸 Frames salvos em: {output_path}")
            return True
        else:
            print(f"\n❌ Erro na renderização (código {process.returncode})")
            return False
            
    except subprocess.TimeoutExpired:
        print("\n⏱️ Timeout - Renderização demorou muito tempo")
        return False
    except Exception as e:
        print(f"\n❌ Erro: {str(e)}")
        return False

print("✅ Função de renderização configurada para UE 4.27!")

## 7. Renderizar Sequência

**Configure os parâmetros abaixo de acordo com seu projeto:**

In [ ]:
# Configurações de Renderização
PROJECT_FILE = f'{UE4_PROJECT_PATH}/MyProject.uproject'  # Ajuste o nome do seu projeto
SEQUENCE = '/Game/Cinematics/MySequence'  # Caminho da sequência no projeto
RENDER_OUTPUT = f'{OUTPUT_PATH}/Render_001'

# Configurações de qualidade
RESOLUTION = '1920x1080'  # Ou '3840x2160' para 4K
FRAME_RATE = 30
FORMAT = 'png'  # png, jpg, ou exr
QUALITY = 100  # 0-100

# Método de renderização
USE_MOVIE_RENDER_QUEUE = False  # True = Movie Render Queue (melhor), False = Legacy

print("=" * 60)
print("VERIFICANDO CONFIGURAÇÃO")
print("=" * 60)
print(f"📁 Projeto: {PROJECT_FILE}")
print(f"🎮 Engine: {UE4_EXECUTABLE}")
print(f"🎞️  Sequência: {SEQUENCE}")
print(f"💾 Output: {RENDER_OUTPUT}")
print("=" * 60 + "\n")

# Verificar se arquivos existem
if not os.path.exists(UE4_EXECUTABLE):
    print(f"❌ Engine não encontrado: {UE4_EXECUTABLE}")
    print("⚠️  Execute as células de compilação (5.0 - 5.6) primeiro!")
elif not os.path.exists(PROJECT_FILE):
    print(f"⚠️ Projeto não encontrado: {PROJECT_FILE}")
    print("Por favor:")
    print("1. Faça upload do seu projeto .uproject para o Google Drive")
    print(f"2. Coloque em: {UE4_PROJECT_PATH}")
    print("3. Ajuste o caminho PROJECT_FILE acima")
    print("\n📋 Arquivos .uproject disponíveis:")
    !find {DRIVE_PATH} -name "*.uproject" 2>/dev/null | head -n 10
else:
    print("✅ Todos os arquivos encontrados!")
    print("\n🎬 Iniciando renderização...\n")
    
    # Executar renderização
    success = render_ue4_sequence(
        project_path=PROJECT_FILE,
        sequence_path=SEQUENCE,
        output_path=RENDER_OUTPUT,
        resolution=RESOLUTION,
        frame_rate=FRAME_RATE,
        output_format=FORMAT,
        quality=QUALITY,
        use_movie_render_queue=USE_MOVIE_RENDER_QUEUE
    )
    
    if success:
        print("\n🎉 Renderização finalizada!")
        print(f"\n📂 Conteúdo do diretório:")
        !ls -lh {RENDER_OUTPUT}

## 8. Converter Frames para Vídeo (Opcional)

In [ ]:
# Instalar FFmpeg
!apt-get install -y ffmpeg

# Converter sequência de imagens para vídeo
def frames_to_video(frames_path, output_video, frame_rate=30, codec='libx264', quality='high'):
    """
    Converte uma sequência de frames em vídeo
    
    Args:
        frames_path: Caminho para os frames (ex: /path/frame_%04d.png)
        output_video: Nome do vídeo de saída
        frame_rate: FPS do vídeo
        codec: Codec de vídeo (libx264, libx265, prores)
        quality: Qualidade (high, medium, low)
    """
    
    # Mapear qualidade para CRF (lower = better quality)
    crf_map = {'high': 18, 'medium': 23, 'low': 28}
    crf = crf_map.get(quality, 23)
    
    cmd = [
        'ffmpeg',
        '-framerate', str(frame_rate),
        '-i', frames_path,
        '-c:v', codec,
        '-crf', str(crf),
        '-pix_fmt', 'yuv420p',
        '-y',  # Sobrescrever se existir
        output_video
    ]
    
    print(f"Convertendo frames para vídeo...")
    print(f"Codec: {codec}, Qualidade: {quality} (CRF {crf})")
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"✅ Vídeo criado: {output_video}")
        return True
    else:
        print(f"❌ Erro ao criar vídeo: {result.stderr}")
        return False

# Exemplo de uso:
# Ajuste o padrão de arquivos conforme a saída do UE4
FRAMES_PATTERN = f'{RENDER_OUTPUT}/frame_%04d.png'
OUTPUT_VIDEO = f'{OUTPUT_PATH}/animation.mp4'

# Descomente para executar:
# frames_to_video(FRAMES_PATTERN, OUTPUT_VIDEO, frame_rate=30, quality='high')

print("Função de conversão configurada!")

## 9. Visualizar Resultados

In [ ]:
# Visualizar alguns frames renderizados
from IPython.display import Image, display
import glob

# Listar frames renderizados
frames = sorted(glob.glob(f'{RENDER_OUTPUT}/*.png'))[:5]  # Primeiros 5 frames

if frames:
    print(f"Visualizando {len(frames)} primeiros frames:\n")
    for frame in frames:
        print(f"Frame: {os.path.basename(frame)}")
        display(Image(filename=frame, width=800))
else:
    print("Nenhum frame encontrado para visualizar.")
    print(f"Verifique o diretório: {RENDER_OUTPUT}")

## 10. Limpeza e Otimização

In [ ]:
# Verificar uso de disco
!df -h

# Limpar arquivos temporários se necessário
# !rm -rf /content/UE4/Intermediate
# !rm -rf /content/UE4/Saved/Logs

print("Espaço em disco verificado!")

## 📝 Notas Importantes

### Preparação do GitHub Token:
1. **Vincular conta GitHub à Epic Games**:
   - Acesse: https://www.unrealengine.com/account/connections
   - Conecte sua conta GitHub
   - Aceite o convite para o repositório EpicGames/UnrealEngine

2. **Criar Personal Access Token**:
   - GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic)
   - Generate new token (classic)
   - Selecione escopo: `repo` (Full control of private repositories)
   - Copie o token gerado (não será mostrado novamente!)

### Compilação do UE 4.27-Chaos:
1. **Tempo estimado**:
   - Clone: 30-60 minutos (~15GB)
   - Setup: 10-15 minutos
   - Compilação: 2-4 horas (depende do CPU)

2. **Otimizações**:
   - Use `--depth 1` no git clone (já configurado)
   - Compile apenas UnrealEditor (não precisa de todo o engine)
   - Monitore espaço em disco (precisa ~60GB)

3. **Alternativa mais rápida**:
   - Compile o UE4 localmente (Linux)
   - Crie um tarball: `tar -czf UE4_Linux.tar.gz UnrealEngine/`
   - Upload para Google Drive
   - Extraia no Colab (muito mais rápido que compilar)

### Preparação do Projeto UE4:
1. **No seu PC** (Windows/Mac/Linux):
   - Abra seu projeto no Unreal Engine 4.27
   - Configure sequências de renderização (Sequencer ou Movie Render Queue)
   - File → Package Project → Linux (ou apenas copie o .uproject)
   - Upload para Google Drive

2. **Estrutura recomendada no Drive**:
   ```
   MyDrive/
   ├── UE4_Projects/
   │   ├── MeuProjeto/
   │   │   ├── MeuProjeto.uproject
   │   │   ├── Content/
   │   │   └── Config/
   └── UE4_Renders/  (output)
   ```

### Movie Render Queue vs Legacy Sequencer:
- **Movie Render Queue** (Recomendado para UE 4.27+):
  - Melhor qualidade
  - Mais controle (anti-aliasing, motion blur, etc)
  - Requer configuração de preset
  
- **Legacy Sequencer**:
  - Mais simples
  - Menos opções
  - Funciona sem configuração adicional

### Limitações do Colab:
- **Tempo máximo**: 12 horas (considere fazer cache do engine compilado)
- **Espaço em disco**: ~100GB (UE4 source ~15GB, compilado ~50GB)
- **RAM**: ~12GB (pode ser limitante para projetos grandes)
- **GPU T4**: 16GB VRAM (bom para renderização 1080p/4K)

### Otimizações de Renderização:
- Use `-MovieQuality=75` para preview rápido
- Renderize em 1080p primeiro, depois 4K se necessário
- Ative `-NoTextureStreaming` para melhor qualidade
- Para animações longas, renderize em batches

### Recursos Úteis:
- [UE4.27 Release Notes](https://docs.unrealengine.com/4.27/en-US/WhatsNew/Builds/ReleaseNotes/4_27/)
- [Movie Render Queue](https://docs.unrealengine.com/4.27/en-US/AnimatingObjects/Sequencer/Workflow/RenderingCmdLine/HighQualityMediaExport/)
- [Command Line Arguments](https://docs.unrealengine.com/4.27/en-US/ProductionPipelines/CommandLineArguments/)
- [Linux Development](https://docs.unrealengine.com/4.27/en-US/SharingAndReleasing/Linux/BeginnerLinuxDeveloper/)

### Solução de Problemas:
- **"Permission denied" no git clone**: Verifique se o token tem permissão `repo`
- **"Repository not found"**: Conta GitHub não vinculada à Epic Games
- **Compilação falha**: Verifique se tem RAM suficiente (feche outras abas)
- **Renderização travada**: Aumente timeout ou use projeto mais leve